In [1]:
"""
Hierarchical Space Themed RPG Content Generator using LangChain with Ollama
This script demonstrates chaining LLM calls to generate:
1. Planet
2. Regions
3. Towns
4. Alien NPCs

Uses JSON output for reliable parsing.

Requirements:
- Ollama installed and running (https://ollama.ai)
- Mistral model pulled: ollama pull mistral
"""


from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import json
import re
from typing import Dict, List
import os
from datetime import datetime

In [2]:
# LLM Configuration
LLM_MODEL = "mistral"
LLM_TEMPERATURE = 0.8 # Can be changed to change model accuracy and fluidity of answers

# File Configuration
OUTPUT_DIR = "."  # Directory to save worlds (current directory by default)

In [3]:
# Initialize the LLM with Ollama
# Make sure Ollama is running and you have pulled mistral: ollama pull mistral
llm = Ollama(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

print("Done")

Done


In [4]:
# ============================================================================
# PROMPT TEMPLATES
# ============================================================================

PLANET_PROMPT = """You are a sci-fi space creator. Generate a unique planet.

Return your response as valid JSON with this exact structure:
{{
  "name": "Planet Name Here",
  "theme": "Sci-Fi/Dystopian/Cyberpunk/Etc",
  "description": "2-3 sentence description of the planets's key features, climate, and technological level",
  "num_regions": 3,
  "conflict": "main tension or conflict that drives the planet's story"
}}

Be creative and specific! Return ONLY valid JSON, no other text."""


REGION_PROMPT = """You are creating kingdoms for the planet "{planet_name}".

Planet Context: {planet_description}
Planet Theme: {planet_theme}
Planet Conflict: {planet_conflict}

Generate exactly 3 regions as a JSON array:

[
  {{
    "name": "First Region Name",
    "leader": "Leader Name and Title",
    "alignment": "good",
    "specialty": "rare metals"
    "description": "Two to three sentences about this region's culture, geography, race that lives there and role in the planet conflict."
  }},
  {{
    "name": "Second Region Name",
    "leader": "Leader Name and Title",
    "alignment": "neutral",
    "specialty": "trade",
    "description": "Two to three sentences about this region's culture, geography, race that lives there and role in the planet conflict."
  }},
  {{
    "name": "Third Region Name",
    "leader": "Leader Name and Title",
    "alignment": "evil",
    "specialty": "warfare",
    "description": "Two to three sentences about this region's culture, geography, race that lives there and role in the planet conflict."
  }}
]

IMPORTANT: Return ONLY the JSON array. Start with [ and end with ]. No extra text."""


TOWN_PROMPT = """You are creating towns for the region "{region_name}" on the planet named {planet_name}.

Region Context: {region_description}
Region Specialty: {region_specialty}
Region Alignment: {region_alignment}

Generate exactly 3 towns as a JSON array:

[
  {{
    "name": "First Town Name",
    "size": "large",
    "industry": "primary industry",
    "feature": "unique landmark",
    "description": "Two sentences about the town."
  }},
  {{
    "name": "Second Town Name",
    "size": "medium",
    "industry": "primary industry",
    "feature": "unique landmark",
    "description": "Two sentences about the town."
  }},
  {{
    "name": "Third Town Name",
    "size": "small",
    "industry": "primary industry",
    "feature": "unique landmark",
    "description": "Two sentences about the town."
  }}
]

IMPORTANT: Return ONLY the JSON array. Start with [ and end with ]. No extra text."""


NPC_PROMPT = """You are creating alien NPCs for the town "{town_name}" in {region_name}, {planet_name}.

Town Context: {town_description}
Town Size: {town_size}
Town Industry: {town_industry}

Generate exactly 4 NPCs as a JSON array:

[
  {{
    "name": "First NPC Name",
    "role": "occupation",
    "race": "race",
    "age": "age",
    "personality": "traits",
    "secret": "hidden quest hook",
    "dialogue_style": "formal",
    "description": "Two sentences."
  }},
  {{
    "name": "Second NPC Name",
    "role": "occupation",
    "race": "race",
    "age": "age",
    "personality": "traits",
    "secret": "hidden quest hook",
    "dialogue_style": "jovial",
    "description": "Two sentences."
  }},
  {{
    "name": "Third NPC Name",
    "role": "occupation",
    "race": "race",
    "age": "age",
    "personality": "traits",
    "secret": "hidden quest hook",
    "dialogue_style": "mysterious",
    "description": "Two sentences."
  }},
  {{
    "name": "Fourth NPC Name",
    "role": "occupation",
    "race": "race",
    "age": "age",
    "personality": "traits",
    "secret": "hidden quest hook",
    "dialogue_style": "casual",
    "description": "Two sentences."
  }}
]

IMPORTANT: Return ONLY the JSON array. Start with [ and end with ]. No extra text."""

In [5]:
# ============================================================================
# JSON PARSING FUNCTIONS
# ============================================================================

def extract_json(text: str) -> str:
    """Extract JSON from LLM output by finding matching brackets."""
    text = text.strip()
    
    # Look for JSON array first (most common)
    arr_start = text.find('[')
    if arr_start != -1:
        bracket_count = 0
        for i in range(arr_start, len(text)):
            if text[i] == '[':
                bracket_count += 1
            elif text[i] == ']':
                bracket_count -= 1
                if bracket_count == 0:
                    return text[arr_start:i+1]
    
    # Look for JSON object
    obj_start = text.find('{')
    if obj_start != -1:
        brace_count = 0
        for i in range(obj_start, len(text)):
            if text[i] == '{':
                brace_count += 1
            elif text[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    return text[obj_start:i+1]
    
    return text


def parse_planet(planet_text: str) -> Dict:
    """Parse planet JSON output."""
    try:
        json_str = extract_json(planet_text)
        planet_data = json.loads(json_str)
        if 'num_regions' in planet_data:
            planet_data['num_regions'] = str(planet_data['num_regions'])
        return planet_data
    except json.JSONDecodeError as e:
        print(f"  JSON parsing error in planet: {e}")
        print(f"Raw output: {planet_text[:300]}")
        return {
            'name': 'Unknown Planet',
            'theme': 'sci-fi',
            'description': 'A mysterious planet',
            'num_regions': '3',
            'conflict': 'Unknown conflict'
        }


def parse_regions(region_text: str) -> List[Dict]:
    """Parse regions JSON array."""
    try:
        json_str = extract_json(region_text)
        regions = json.loads(json_str)
        if not isinstance(regions, list):
            regions = [regions]
        return regions
    except json.JSONDecodeError as e:
        print(f"  JSON parsing error in regions: {e}")
        print(f"Raw output: {region_text[:300]}")
        return []


def parse_towns(town_text: str) -> List[Dict]:
    """Parse towns JSON array."""
    try:
        json_str = extract_json(town_text)
        towns = json.loads(json_str)
        if not isinstance(towns, list):
            towns = [towns]
        return towns
    except json.JSONDecodeError as e:
        print(f"  JSON parsing error in towns: {e}")
        print(f"Raw output: {town_text[:300]}")
        return []


def parse_npcs(npc_text: str) -> List[Dict]:
    """Parse NPCs JSON array."""
    try:
        json_str = extract_json(npc_text)
        npcs = json.loads(json_str)
        if not isinstance(npcs, list):
            npcs = [npcs]
        return npcs
    except json.JSONDecodeError as e:
        print(f"  JSON parsing error in NPCs: {e}")
        print(f"Raw output: {npc_text[:300]}")
        return []

In [6]:
# ============================================================================
# WORLD GENERATOR CLASS
# ============================================================================

class RPGPlanetGenerator:
    """
    Hierarchical Space themed RPG world generator.
    Generates: World → Regions (3) → Towns (3 each) → NPCs (4 each)
    """
    
    def __init__(self, llm):
        self.llm = llm
        
        # Create LangChain chains for each level
        self.planet_chain = LLMChain(
            llm=llm,
            prompt=PromptTemplate(template=PLANET_PROMPT, input_variables=[])
        )
        
        self.region_chain = LLMChain(
            llm=llm,
            prompt=PromptTemplate(
                template=REGION_PROMPT,
                input_variables=["planet_name", "planet_description", "planet_theme", "planet_conflict"]
            )
        )
        
        self.town_chain = LLMChain(
            llm=llm,
            prompt=PromptTemplate(
                template=TOWN_PROMPT,
                input_variables=["region_name", "region_description", 
                               "region_specialty", "region_alignment", "planet_name"]
            )
        )
        
        self.npc_chain = LLMChain(
            llm=llm,
            prompt=PromptTemplate(
                template=NPC_PROMPT,
                input_variables=["town_name", "region_name", "planet_name",
                               "town_description", "town_size", "town_industry"]
            )
        )
        
    def generate_planet(self):
        """Step 1: Generate the planet."""
        print(" Generating planet...")
        planet_text = self.planet_chain.invoke({})['text']
        planet_data = parse_planet(planet_text)
        print(f"Created planet: {planet_data.get('name', 'Unknown')}")
        return planet_data
    
    def generate_regions(self, planet_data):
        """Step 2: Generate 3 regions."""
        print(f"\n\t Generating 3 regions...")
        
        region_text = self.region_chain.invoke({
            "planet_name": planet_data['name'],
            "planet_description": planet_data['description'],
            "planet_theme": planet_data['theme'],
            "planet_conflict": planet_data['conflict']
        })['text']
        
        regions = parse_regions(region_text)
        print(f"\t Created {len(regions)} regions")
        return regions
    
    def generate_towns(self, planet_data, region):
        """Step 3: Generate 3 towns for a region."""
        print(f" \t\t Generating towns for {region['name']}...")
        
        town_text = self.town_chain.invoke({
            "region_name": region['name'],
            "region_description": region['description'],
            "region_specialty": region['specialty'],
            "region_alignment": region['alignment'],
            "planet_name": planet_data['name']
        })['text']
        
        towns = parse_towns(town_text)
        print(f"  \t\tCreated {len(towns)} towns")
        return towns
    
    def generate_npcs(self, planet_data, region, town):
        """Step 4: Generate 4 NPCs for a town."""
        print(f" \t\t\tGenerating NPCs for {town['name']}...")
        
        npc_text = self.npc_chain.invoke({
            "town_name": town['name'],
            "region_name": region['name'],
            "planet_name": planet_data['name'],
            "town_description": town['description'],
            "town_size": town['size'],
            "town_industry": town['industry']
        })['text']
        
        npcs = parse_npcs(npc_text)
        print(f" \t\t\tCreated {len(npcs)} NPCs")
        return npcs
    
    def generate_full_planet(self):
        """Generate complete planet hierarchy."""
        print("=" * 60)
        print("STARTING PLANET GENERATION")
        print("=" * 60)
        
        # Generate planet
        planet_data = self.generate_planet()
        
        # Generate kingdoms
        regions = self.generate_regions(planet_data)
        
        # Generate towns and NPCs
        for region in regions:
            towns = self.generate_towns(planet_data, region)
            region['towns'] = towns
            
            for town in towns:
                npcs = self.generate_npcs(planet_data, region, town)
                town['npcs'] = npcs
        
        print("\n" + "=" * 60)
        print("PLANET GENERATION COMPLETE!")
        print("=" * 60)
        
        return {
            'planet': planet_data,
            'regions': regions
        }

In [7]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def print_planet_summary(planet_data):
    """Print a summary of the generated planet."""
    planet = planet_data['planet']
    print(f"\n  PLANET: {planet['name']}")
    print(f"Theme: {planet['theme']}")
    print(f"Conflict: {planet['conflict']}")
    
    for region in planet_data['regions']:
        print(f"\n \t\t{region['name']} - {region['alignment']}")
        for town in region.get('towns', []):
            print(f" \t\t\t {town['name']} ({town['size']})")
            for npc in town.get('npcs', []):
                print(f" \t\t\t\t {npc['name']} - {npc['role']}  -  {npc['race']}")


def save_planet_to_json(planet_data, filename=None):
    """Save planet to JSON file."""
    if filename is None:
        planet_name = planet_data['planet'].get('name', 'Unknown_Planet')
        clean_name = re.sub(r'[^\w\s-]', '', planet_name).strip().replace(' ', '_')
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"planet_{clean_name}_{timestamp}.json"
    
    save_data = {
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'generator_version': '3.0',
            'model': 'mistral',
            'structure': '3 regions, 3 towns each, 4 NPCs each'
        },
        'planet_data': planet_data
    }
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, indent=2, ensure_ascii=False)
    
    print(f"\n  Planet saved to: {filename}")
    return filename


def load_planet_from_json(filename):
    """Load planet from JSON file."""
    with open(filename, 'r', encoding='utf-8') as f:
        save_data = json.load(f)
    return save_data.get('planet_data', save_data)

def get_planet_stats(planet_data):
    """Calculate planet statistics."""
    num_regions = len(planet_data['regions'])
    num_towns = sum(len(k.get('towns', [])) for k in planet_data['regions'])
    num_npcs = sum(
        len(t.get('npcs', [])) 
        for k in planet_data['regions'] 
        for t in k.get('towns', [])
    )
    
    return {
        'planet_name': planet_data['planet'].get('name', 'Unknown'),
        'num_regions': num_regions,
        'num_towns': num_towns,
        'num_npcs': num_npcs,
        'total_entities': num_regions + num_towns + num_npcs
    }


def print_planet_stats(planet_data):
    """Print planet statistics."""
    stats = get_planet_stats(planet_data)
    print("\n" + "=" * 60)
    print("PLANET STATISTICS")
    print("=" * 60)
    print(f"Planet: {stats['planet_name']}")
    print(f"Regions: {stats['num_regions']}")
    print(f"Towns: {stats['num_towns']}")
    print(f"NPCs: {stats['num_npcs']}")
    print(f"Total Entities: {stats['total_entities']}")
    print("=" * 60)

In [8]:
# Generate planet
generator = RPGPlanetGenerator(llm)
planet_data = generator.generate_full_planet()

# Display results
print_planet_summary(planet_data)
print_planet_stats(planet_data)

# Save to file
saved_filename = save_planet_to_json(planet_data)
print(f"\nGeneration complete! Saved to {saved_filename}")

C:\Users\hb536\AppData\Local\Temp\ipykernel_43160\3310233964.py:15: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  self.planet_chain = LLMChain(


STARTING PLANET GENERATION
 Generating planet...
Created planet: Zyxon-9

	 Generating 3 regions...


KeyboardInterrupt: 